# SPY Data Exploration

Analysis of SPY daily OHLCV data from 2015-2024 using the project's DataFetcher and DataStore.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy import stats

from src.data.fetcher import YFinanceFetcher
from src.data.store import DataStore
from src.data import df_to_candles

In [ ]:
from datetime import datetime

fetcher = YFinanceFetcher()
store = DataStore(cache_dir="../data/raw")
df = store.fetch_or_cache("SPY", datetime(2015, 1, 1), datetime(2024, 12, 31), fetcher)
print(f"Shape: {df.shape}")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
df.tail()

## Candlestick Chart

In [ ]:
recent = df.tail(252)
fig = go.Figure(go.Candlestick(
    x=recent.index,
    open=recent["open"], high=recent["high"],
    low=recent["low"], close=recent["close"],
    name="SPY"
))
fig.update_layout(title="SPY — Last 252 Trading Days", xaxis_title="Date", yaxis_title="Price ($)", height=500)
fig.show()

## Volume Analysis

In [ ]:
fig = go.Figure(go.Bar(x=df.index, y=df["volume"], name="Volume", marker_color="steelblue", opacity=0.7))
fig.update_layout(title="SPY Daily Volume (2015-2024)", xaxis_title="Date", yaxis_title="Volume", height=350)
fig.show()

## Return Distribution

In [ ]:
returns = df["close"].pct_change().dropna()
print(f"Mean daily return:  {returns.mean()*100:.4f}%")
print(f"Std daily return:   {returns.std()*100:.4f}%")
print(f"Skewness:           {returns.skew():.4f}")
print(f"Kurtosis (excess):  {returns.kurtosis():.4f}")
print(f"Ann. return (approx): {((1+returns.mean())**252 - 1)*100:.1f}%")
print(f"Ann. volatility:    {returns.std()*np.sqrt(252)*100:.1f}%")

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Return Distribution", "Q-Q Plot vs Normal"])

fig.add_trace(go.Histogram(x=returns, nbinsx=80, name="Returns", marker_color="steelblue", opacity=0.7), row=1, col=1)

# Q-Q plot
(osm, osr), (slope, intercept, r) = stats.probplot(returns)
fig.add_trace(go.Scatter(x=list(osm), y=list(osr), mode="markers", name="Q-Q", marker=dict(size=3, color="steelblue")), row=1, col=2)
x_line = np.array([min(osm), max(osm)])
fig.add_trace(go.Scatter(x=list(x_line), y=list(slope*x_line+intercept), mode="lines", name="Normal", line=dict(color="red")), row=1, col=2)

fig.update_layout(height=400, showlegend=False, title="SPY Daily Returns Analysis")
fig.show()